### Import packages

In [1]:
import os
from openai import OpenAI
import pandas as pd
import json
from tqdm.auto import tqdm

### Check if environment variable is set for OpenAI API Key

In [2]:
try:
    api_key = os.environ["OPENAI_API_KEY"]
except KeyError as exc:
    raise RuntimeError("Environment variable OPENAI_API_KEY missing.") from exc

### Define which OpenAI model to use

In [3]:
MODEL = 'gpt-5-nano'

### Construct OpenAI client instance

In [4]:
client = OpenAI(api_key=api_key)

def send_LLM(prompt: str) -> str:
    try:
        # Send chat request
        chat_completion = client.chat.completions.create(
            model = MODEL,
            messages=[
                {"role": "system", "content": "You are a rigorous JSON-only annotator of online irony."},
                {"role": "user", "content": prompt}
            ])
    except KeyError as exc:
        raise RuntimeError("Error") from exc

    return chat_completion.choices[0].message.content

### Define schema for irony as JSON

In [5]:
IRONY_SCHEMA = {
    "irony": {
        "irony_score": "1-5 integer (1 = not ironic, 5 = very ironic)",
        "confidence_score": "1-5 integer (1 = not confident, 5 = very confident)",
        "justification": "string - explanation based on observed language",
        "irony_markers": ["string"]
    } 
}

def prompt_irony_analysis(thread_title: str, comment: str) -> str:
    schema_str = json.dumps(IRONY_SCHEMA, indent=2)
    return f"""
You are a rigorous JSON-only annotator of online irony.

Your task is to analyze whether a given Reddit comment contains irony.
The comments come from the “r/de” subreddit.
The goal is to annotate comments based on the definition and use the results to provide a foundation for a scientific analysis. 

Definition:  

Irony is a rhetorical device in which a statement deliberately creates a discrepancy between its literal meaning and its intended meaning.  

Analyze the COMMENT under the THREAD TITLE below.

Return ONLY one JSON object that strictly matches this schema:

{schema_str}

THREAD TITLE:
\"\"\"{thread_title}\"\"\"

COMMENT:
\"\"\"{comment}\"\"\"

"""

### Read input data

In [6]:
df = pd.read_csv('dataset.csv')

df.head()

,collected_by,id,thread_title,comment
0,Johan,1,Mehrheit sieht ältere im Vorteil – nicht einma...,Solange alle ihre Tiefkühlpizza und RTL haben ...
1,Johan,2,Mehrheit sieht ältere im Vorteil – nicht einma...,Ich bin schockiert.
2,Johan,3,Mehrheit sieht ältere im Vorteil – nicht einma...,"Würden sich die babyboomer zu Tode arbeiten, b..."
3,Johan,4,Mehrheit sieht ältere im Vorteil – nicht einma...,Auf den Schreck erstmal eine Rentenerhöhung.
4,Johan,5,Mehrheit sieht ältere im Vorteil – nicht einma...,Finde den Generationen-Vertrag an sich schon d...


In [7]:
df.shape

(60, 4)

### Perform LLM irony analysis on every comment
We had 9 human annotators and as LLMs are non-deterministic, we also perform it 9 times to see potential variability.

In [ ]:
results = []
num_of_iterations = 9

for iteration in tqdm(range(num_of_iterations), desc="LLM iterations", position=0):
    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Processing comments",
        position=1,
        leave=False
    ):  
        row_id = row["id"]
        thread_title = row["thread_title"]
        comment = row["comment"]

        prompt = prompt_irony_analysis(thread_title, comment)
        llm_answer = send_LLM(prompt)

        # Parse JSON safely
        try:
            parsed = json.loads(llm_answer)
            irony_score = parsed.get("irony_score", None)
            confidence_score = parsed.get("confidence_score", None)
        except json.JSONDecodeError:
            irony_score = None
            confidence_score = None

        results.append({
            "id": row_id,
            "iteration": iteration,
            "thread_title": thread_title,
            "comment": comment,
            "irony_score": irony_score,
            "confidence_score": confidence_score
        })

# Convert to DataFrame
llm_df = pd.DataFrame(results)

# Save as CSV
llm_df.to_csv("llm_results.csv", index=False)

LLM iterations:   0%|          | 0/9 [00:00<?, ?it/s]

Processing comments:   0%|          | 0/60 [00:00<?, ?it/s]

Processing comments:   0%|          | 0/60 [00:00<?, ?it/s]

Processing comments:   0%|          | 0/60 [00:00<?, ?it/s]

Processing comments:   0%|          | 0/60 [00:00<?, ?it/s]

Processing comments:   0%|          | 0/60 [00:00<?, ?it/s]